<p style="align: center;"><img align=center src="https://drive.google.com/uc?export=view&id=1I8kDikouqpH4hf7JBiSYAeNT2IO52T-T" width=600 height=480/></p>
<h3 style="text-align: center;"><b>Школа глубокого обучения ФПМИ МФТИ</b></h3>

<h3 style="text-align: center;"><b>Домашнее задание. Generative adversarial networks</b></h3>



В этом домашнем задании вы обучите GAN генерировать лица людей и посмотрите на то, как можно оценивать качество генерации

In [ ]:
import os
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import torchvision.transforms as tt
import torch
import torch.nn as nn
import cv2
from tqdm.notebook import tqdm
from torchvision.utils import save_image
from torchvision.utils import make_grid
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

sns.set(style='darkgrid', font_scale=1.2)

## Часть 1. Подготовка данных (0.5 балла)

В качестве обучающей выборки возьмем часть датасета [Flickr Faces](https://github.com/NVlabs/ffhq-dataset), который содержит изображения лиц людей в высоком разрешении (1024х1024). Оригинальный датасет очень большой, поэтому мы возьмем его часть. Скачать датасет можно [здесь](https://www.kaggle.com/datasets/tommykamaz/faces-dataset-small?resource=download-directory) и  [здесь](https://drive.google.com/file/d/1inyvLrN5wKBGCxQ4znMKBc64uL4uP_2x/view?usp=drive_link)

Давайте загрузим наши изображения. Напишите функцию, которая строит DataLoader для изображений, при этом меняя их размер до нужного значения (размер 1024 слишком большой, поэтому мы рекомендуем взять размер 128 либо немного больше)

In [ ]:
DATA_DIR = '/kaggle/input/faces-dataset-small' #/content/data/'
image_size = 128
batch_size = 64
stats = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)


In [ ]:
sample_dir = 'generated'
os.makedirs(sample_dir, exist_ok=True)

In [ ]:
def get_dataloader(image_size=image_size, batch_size=batch_size):
  """
  Builds dataloader for training data.
  Use tt.Compose and tt.Resize for transformations
  :param image_size: height and wdith of the image
  :param batch_size: batch_size of the dataloader
  :returns: DataLoader object
  """
  # TODO: resize images, convert them to tensors and build dataloader
  train_ds = ImageFolder(DATA_DIR, transform=tt.Compose([
    tt.Resize(image_size),
    tt.CenterCrop(image_size),
    tt.ToTensor(),
    tt.Normalize(*stats)
    ]))

  train_dl = DataLoader(train_ds, batch_size, shuffle=True, num_workers=2, pin_memory=True)
  return train_dl

In [ ]:
def denorm(img_tensors):
    return img_tensors * stats[1][0] + stats[0][0]

In [ ]:
def show_images(images, nmax=64):
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_xticks([]); ax.set_yticks([])
    ax.imshow(make_grid(denorm(images.detach()[:nmax]), nrow=8).permute(1, 2, 0))

def show_batch(dl, nmax=64):
    for images, _ in dl:
        show_images(images, nmax)
        break

In [ ]:
def save_samples(index, latent_tensors, show=True):
    fake_images = generator(latent_tensors)
    fake_fname = 'generated-images-{0:0=4d}.png'.format(index)
    save_image(denorm(fake_images), os.path.join(sample_dir, fake_fname), nrow=8)
    print('Saving', fake_fname)
    if show:
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.set_xticks([]); ax.set_yticks([])
        ax.imshow(make_grid(fake_images.cpu().detach(), nrow=8).permute(1, 2, 0))

In [ ]:
#TODO: build dataloader and transfer it to device
# Зададим девайс: cuda или cpu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
def to_device(data, device):
    """Move tensor(s) to chosen device"""
    if isinstance(data, (list,tuple)):
        return [to_device(x, device) for x in data]
    return data.to(device, non_blocking=True)

class DeviceDataLoader():
    """Wrap a dataloader to move data to a device"""
    def __init__(self, dl, device):
        self.dl = dl
        self.device = device

    def __iter__(self):
        """Yield a batch of data after moving it to device"""
        for b in self.dl:
            yield to_device(b, self.device)

    def __len__(self):
        """Number of batches"""
        return len(self.dl)

In [ ]:
train_dl = get_dataloader()
train_dl = DeviceDataLoader(train_dl, device)


In [ ]:
#batch = next(iter(train_dl))

In [ ]:
#show_images(batch[0][2])

In [ ]:
#show_images(denorm(batch[0][2]))

## Часть 2. Построение и обучение модели (2 балла)

Сконструируйте генератор и дискриминатор. Помните, что:
* дискриминатор принимает на вход изображение (тензор размера `3 x image_size x image_size`) и выдает вероятность того, что изображение настоящее (тензор размера 1)

* генератор принимает на вход тензор шумов размера `latent_size x 1 x 1` и генерирует изображение размера `3 x image_size x image_size`

In [ ]:
discriminator = nn.Sequential(
    # in: 3 x 128 x 128

    nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=2, bias=False),
    nn.BatchNorm2d(64),
    nn.LeakyReLU(0.2, inplace=True),
    # out: 64 x 64 x 64

    nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(128),
    nn.LeakyReLU(0.2, inplace=True),
    # out: 128 x 32 x 32

    nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(256),
    nn.LeakyReLU(0.2, inplace=True),
    # out: 256 x 16 x 16

    nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(512),
    nn.LeakyReLU(0.2, inplace=True),
    # out: 512 x 8 x 8

    nn.Conv2d(512, 1, kernel_size=8, stride=1, padding=0, bias=False),   # out: 512 x 1 x 1
    nn.Flatten(),  # [B, 1, 1, 1] → [B, 1]
    #nn.Sigmoid()
    )

In [ ]:
discriminator = to_device(discriminator, device)

In [ ]:
latent_size = 128 # choose latent size

In [ ]:
fixed_latent = torch.randn(64, latent_size, 1, 1, device=device)

In [ ]:
latent = torch.randn(batch_size, latent_size, 1, 1, device=device)
latent.shape

In [ ]:
#gen_out = generator(latent)
#gen_out.shape

In [ ]:
#dis_out = discriminator(gen_out)
#dis_out

In [ ]:
generator = nn.Sequential(
    # in: latent_size x 1 x 1

    nn.ConvTranspose2d(latent_size, 512, kernel_size=4, stride=1, padding=0, bias=False),
    nn.BatchNorm2d(512),
    nn.ReLU(True),
    # out: 512 x 4 x 4

    nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(256),
    nn.ReLU(True),
    # out: 256 x 8 x 8

    nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(128),
    nn.ReLU(True),
    # out: 128 x 16 x 16

    nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(64),
    nn.ReLU(True),
    # out: 64 x 32 x 32

    nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(32),
    nn.ReLU(True),
    # out: 3 x 64 x 64

    nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1, bias=False),
    nn.Tanh()
    # out: 3 x 128 x 128
)

In [ ]:
generator = to_device(generator, device)

Перейдем теперь к обучению нашего GANа. Алгоритм обучения следующий:
1. Учим дискриминатор:
  * берем реальные изображения и присваиваем им метку 1
  * генерируем изображения генератором и присваиваем им метку 0
  * обучаем классификатор на два класса

2. Учим генератор:
  * генерируем изображения генератором и присваиваем им метку 0
  * предсказываем дискриминаторором, реальное это изображение или нет


В качестве функции потерь берем бинарную кросс-энтропию

In [ ]:
def fit(model, criterion, epochs, lr, start_idx=1):
  # TODO: build optimizers and train your GAN
   # Losses & scores
   losses_g = []
   losses_d = []
   real_scores = []
   fake_scores = []

  # Create optimizers
   optimizer = {
        "discriminator": torch.optim.Adam(model["discriminator"].parameters(),
                                          lr=lr, betas=(0.5, 0.999)),
        "generator": torch.optim.Adam(model["generator"].parameters(),
                                      lr=lr*2, betas=(0.5, 0.999))
   }

   for epoch in range(epochs):
        loss_d_per_epoch = []
        loss_g_per_epoch = []
        real_score_per_epoch = []
        fake_score_per_epoch = []
        for real_images, _ in tqdm(train_dl):
            # Train discriminator
            # Clear discriminator gradients
            optimizer["discriminator"].zero_grad()

             # Pass real images through discriminator
            real_preds = model["discriminator"](real_images)
            real_targets = torch.full((real_images.size(0), 1), 0.85, device=device)

            #real_targets = torch.ones(real_images.size(0), 1, device=device)
            #print((real_preds.shape, real_targets.shape))
            real_loss = criterion["discriminator"](real_preds, real_targets)
            cur_real_score = torch.mean(real_preds).item()

            # Generate fake images
            latent = torch.randn(batch_size, latent_size, 1, 1, device=device)
            fake_images = model["generator"](latent)
            #show_images(fake_images)

            # Pass fake images through discriminator
            fake_targets = torch.zeros(fake_images.size(0), 1, device=device)

            fake_preds = model["discriminator"](fake_images)
            fake_loss = criterion["discriminator"](fake_preds, fake_targets)
            cur_fake_score = torch.mean(fake_preds).item()

            real_score_per_epoch.append(cur_real_score)
            fake_score_per_epoch.append(cur_fake_score)

             # Update discriminator weights
            loss_d = real_loss + fake_loss
            loss_d.backward()
            optimizer["discriminator"].step()
            loss_d_per_epoch.append(loss_d.item())

            # Train generator
            # Clear generator gradients
            optimizer["generator"].zero_grad()

            # Generate fake images
            latent = torch.randn(batch_size, latent_size, 1, 1, device=device)
            fake_images = model["generator"](latent)

            # Try to fool the discriminator
            preds = model["discriminator"](fake_images)
            targets = torch.ones(batch_size, 1, device=device)
            loss_g = criterion["generator"](preds, targets)

            # Update generator weights
            loss_g.backward()
            optimizer["generator"].step()
            loss_g_per_epoch.append(loss_g.item())

        # Record losses & scores
        losses_g.append(np.mean(loss_g_per_epoch))
        losses_d.append(np.mean(loss_d_per_epoch))
        real_scores.append(np.mean(real_score_per_epoch))
        fake_scores.append(np.mean(fake_score_per_epoch))


        # Log losses & scores (last batch)
        print("Epoch [{}/{}], loss_g: {:.4f}, loss_d: {:.4f}, real_score: {:.4f}, fake_score: {:.4f}".format(
            epoch+1, epochs,
            losses_g[-1], losses_d[-1], real_scores[-1], fake_scores[-1]))

        # Save generated images
        if epoch == epochs - 1:
          save_samples(epoch+start_idx, fixed_latent, show=False)

   return losses_g, losses_d, real_scores, fake_scores




In [ ]:
lr = 0.0002

model = {
    "discriminator": discriminator,
    "generator": generator
}

criterion = {
    "discriminator": nn.BCEWithLogitsLoss(),
    "generator": nn.BCEWithLogitsLoss()
}
epochs = 40

In [ ]:
history = fit(model, criterion, epochs, lr)

In [ ]:
losses_g, losses_d, real_scores, fake_scores = history

In [ ]:
generated_img = cv2.imread('/kaggle/working/generated/generated-images-0010.png')
generated_img = generated_img[:, :, [2, 1, 0]]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_xticks([]); ax.set_yticks([])
ax.imshow(generated_img)

Постройте графики лосса для генератора и дискриминатора. Что вы можете сказать про эти графики?

In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(losses_d, '-')
plt.plot(losses_g, '-')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend(['Discriminator', 'Generator'])
plt.title('Losses');

## Часть 3. Генерация изображений

Теперь давайте оценим качество получившихся изображений. Напишите функцию, которая выводит изображения, сгенерированные нашим генератором

In [ ]:
n_images = 4

fixed_latent = torch.randn(n_images, latent_size, 1, 1, device=device)
fake_images = model["generator"](fixed_latent)

In [ ]:
def show_images(images, nmax=64):
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_xticks([]); ax.set_yticks([])
    ax.imshow(make_grid(denorm(images.detach()[:nmax]), nrow=8).permute(1, 2, 0))

In [ ]:
show_images(fake_images.to('cpu'))

Как вам качество получившихся изображений?

Качество  плохое. Попробуем реализовать WGAN

In [ ]:
def fit_wgan(generator, discriminator, dataloader, device, epochs=5, latent_dim=100, lr=5e-5, clip_value=0.01, n_critic=5):
    generator.to(device)
    discriminator.to(device)

    optimizer_G = torch.optim.RMSprop(generator.parameters(), lr=lr*2)
    optimizer_D = torch.optim.RMSprop(discriminator.parameters(), lr=lr)

    losses_g = []
    losses_d = []
    real_scores = []
    fake_scores = []

    for epoch in range(epochs):
        loss_g_epoch = []
        loss_d_epoch = []
        real_score_epoch = []
        fake_score_epoch = []

        for i, (real_imgs, _) in enumerate(tqdm(dataloader)):
            real_imgs = real_imgs.to(device)
            batch_size = real_imgs.size(0)

            # Train Discriminator
            optimizer_D.zero_grad()

            # Sample noise as generator input
            z = torch.randn(batch_size, latent_dim, 1, 1).to(device)
            fake_imgs = generator(z).detach()

            # Compute WGAN loss
            real_out = discriminator(real_imgs)
            fake_out = discriminator(fake_imgs)
            loss_D = -torch.mean(real_out) + torch.mean(fake_out)
            loss_D.backward()
            optimizer_D.step()

            # Weight clipping
            for p in discriminator.parameters():
                p.data.clamp_(-clip_value, clip_value)

            # Save scores
            real_score_epoch.append(real_out.mean().item())
            fake_score_epoch.append(fake_out.mean().item())
            loss_d_epoch.append(loss_D.item())

            # Train the generator every n_critic steps
            if i % n_critic == 0:
                optimizer_G.zero_grad()
                gen_imgs = generator(z)
                loss_G = -torch.mean(discriminator(gen_imgs))
                loss_G.backward()
                optimizer_G.step()
                loss_g_epoch.append(loss_G.item())

        losses_g.append(np.mean(loss_g_epoch))
        losses_d.append(np.mean(loss_d_epoch))
        real_scores.append(np.mean(real_score_epoch))
        fake_scores.append(np.mean(fake_score_epoch))

        print(f"Epoch [{epoch+1}/{epochs}] | Loss_D: {losses_d[-1]:.4f} | Loss_G: {losses_g[-1]:.4f} | Real Score: {real_scores[-1]:.4f} | Fake Score: {fake_scores[-1]:.4f}")

        # Save sample output
        with torch.no_grad():
            sample = generator(torch.randn(64, latent_dim, 1, 1).to(device))
            save_image(sample * 0.5 + 0.5, f"generated_epoch_{epoch+1}.png")  # Rescale to [0, 1]

    print("Training complete.")
    #return losses_g, losses_d, real_scores, fake_scores



In [ ]:

epochs = 40
lr = 0.00004
clip_value = 0.01
n_critic = 5

fit_wgan(
    generator=generator,
    discriminator=discriminator,
    dataloader=train_dl,     
    device=device,           
    epochs=epochs,
    latent_dim=latent_size,
    lr=lr,
    clip_value=clip_value,
    n_critic=n_critic
)

In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(losses_d, '-')
plt.plot(losses_g, '-')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend(['Discriminator', 'Generator'])
plt.title('Losses');

In [ ]:
n_images = 4

fixed_latent = torch.randn(n_images, latent_size, 1, 1, device=device)
fake_images = model["generator"](fixed_latent)

In [ ]:
show_images(fake_images.to('cpu'))

## Часть 4. Leave-one-out-1-NN classifier accuracy (6 баллов)

### 4.1. Подсчет accuracy (1.5 балл)

Не всегда бывает удобно оценивать качество сгенерированных картинок глазами. В качестве альтернативы вам предлагается реализовать следующий подход:
  * Сгенерировать столько же фейковых изображений, сколько есть настоящих в обучающей выборке. Присвоить фейковым метку класса 0, настоящим – 1.
  * Построить leave-one-out оценку: обучить 1NN Classifier (`sklearn.neighbors.KNeighborsClassifier(n_neighbors=1)`) предсказывать класс на всех объектах, кроме одного, проверить качество (accuracy) на оставшемся объекте. В этом вам поможет `sklearn.model_selection.LeaveOneOut`

In [ ]:
train_ds = ImageFolder(DATA_DIR, transform=tt.Compose([
    tt.Resize(image_size),
    tt.CenterCrop(image_size),
    tt.ToTensor(),
    tt.Normalize(*stats)
    ]))

In [ ]:
import torch
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score
import numpy as np

# ⚙ Параметры
sample_size = 500  # оставить меньше для разумного времени LOO

# 1. Собираем реальные изображения на GPU
real_images = torch.stack([train_ds[i][0] for i in range(sample_size)]).to(device)  # [B, C, H, W]

# 2. Генерируем фейковые изображения на GPU
generator.eval()
with torch.no_grad():
    z = torch.randn(sample_size, latent_size, 1, 1, device=device)
    fake_images = generator(z)

# 3. Объединяем данные и перемещаем в numpy один раз
X_real = real_images.view(sample_size, -1).cpu().numpy()
X_fake = fake_images.view(sample_size, -1).cpu().numpy()
X = np.concatenate([X_real, X_fake], axis=0)
y = np.concatenate([np.ones(sample_size), np.zeros(sample_size)])

# 4. Leave-One-Out + 1-NN (CPU)
loo = LeaveOneOut()
knn = KNeighborsClassifier(n_neighbors=1)

y_true = []
y_pred = []

for train_idx, test_idx in loo.split(X):
    knn.fit(X[train_idx], y[train_idx])
    y_true.append(y[test_idx][0])
    y_pred.append(knn.predict(X[test_idx])[0])

acc = accuracy_score(y_true, y_pred)
print(f"Leave-One-Out Accuracy: {acc:.4f}")


Что вы можете сказать о получившемся результате? Какой accuracy мы хотели бы получить и почему?

In [ ]:
Accuracy не близка к 1, это значит, что дискриминатор уже не так хорошо отличает сгенерированнные изображения от реальных. В идеале, хотелось бы получить Accuracy близкую к 0.5. Это бы означало, что генератор хорошо научился обманывать дискриминатор.

### 4.2. Визуализация распределений (1 балл)

Давайте посмотрим на то, насколько похожи распределения настоящих и фейковых изображений. Для этого воспользуйтесь методом, снижающим размерность (к примеру, TSNE) и изобразите на графике разным цветом точки, соответствующие реальным и сгенерированным изображенияи

In [ ]:
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# TSNE для визуализации в 2D
tsne = TSNE(n_components=2, random_state=42, perplexity=30, init="pca")
X_embedded = tsne.fit_transform(X)  # X — из предыдущего блока (объединённый real + fake)

# Разделим координаты по классам
X_real_2d = X_embedded[:sample_size]
X_fake_2d = X_embedded[sample_size:]

# Визуализация
plt.figure(figsize=(8, 6))
plt.scatter(X_real_2d[:, 0], X_real_2d[:, 1], c="green", label="Real", alpha=0.6)
plt.scatter(X_fake_2d[:, 0], X_fake_2d[:, 1], c="red", label="Fake", alpha=0.6)
plt.legend()
plt.title("TSNE: Real vs Fake Image Embeddings")
plt.xlabel("TSNE-1")
plt.ylabel("TSNE-2")
plt.grid(True)
plt.show()


Прокомментируйте получившийся результат:

Реальные и сгенерированные изображения сильно разделены. Ожидаемый результат при хорошо обученной модели GAN - Реальные и сгенерированные изображения перемешаны в проекции на двумерное пространство.